# Generate S3 traffic for InstantEvidence

Use this notebook to create S3 activity in your bucket—uploads, reads, copies, listings, and deletes—so you can confirm **InstantEvidence** is receiving and evaluating events the way you expect.

Operations run through the [AWS API MCP Server](https://github.com/awslabs/mcp/tree/main/src/aws-api-mcp-server), which executes standard AWS S3 API calls using the credentials you provide.

Object keys use mixed prefixes (for example `public/`, `data/`, and `secret/`) so InstantEvidence can exercise allow/deny style rules during demos.

## Before you begin

### AWS credentials

In Google Colab, open **Secrets** (key icon in the left sidebar) and add:

| Secret | Required |
|--------|----------|
| `AWS_ACCESS_KEY_ID` | Yes |
| `AWS_SECRET_ACCESS_KEY` | Yes |
| `AWS_REGION` | No — defaults to `us-east-1` |
| `AWS_SESSION_TOKEN` | No — only if your credentials include a session token (SSO or assumed role) |

Use credentials scoped to your **test bucket only**. Minimum IAM actions:

- `sts:GetCallerIdentity`
- `s3:HeadBucket`, `s3:ListBucket` on the bucket
- `s3:PutObject`, `s3:GetObject`, `s3:HeadObject`, `s3:DeleteObject`, `s3:CopyObject` on `arn:aws:s3:::YOUR_BUCKET/*`

Avoid production admin keys.

### Your bucket and InstantEvidence

- Set **BUCKET** in the run cell to the same S3 bucket InstantEvidence monitors.
- Ensure object events from that bucket reach InstantEvidence (for example via SNS or EventBridge to your S3 event webhook).

### Open from this repo

Open the notebook from the [traffic-generator](https://github.com/synapse6-ai/traffic-generator) repo in Colab so the helper modules in `colab/` load automatically:

`aws_credentials.py` → `mcp_s3_client.py` → `traffic_generator.py`

If you upload only the `.ipynb`, upload those three files to the same Colab directory when prompted.


### Run order (important in Colab)

Run cells top to bottom. The **load modules** and **import** cells must finish before the **Run traffic** cell. If MCP connection fails, use **Runtime → Restart session**, then **Run all** again.

## How to run

1. Open this notebook in [Google Colab](https://colab.research.google.com/) from GitHub (recommended) or upload the `colab/` files together.
2. Add the secrets above.
3. Set **BUCKET** to your bucket name and adjust **CYCLES** / intervals if needed.
4. Select **Runtime → Run all**.

The traffic loop usually takes a few minutes (default 20 cycles with pauses between operations). If it stops early after repeated errors, check the bucket name, secrets, and IAM permissions.



In [ ]:
%pip install -q awslabs.aws-api-mcp-server mcp


In [ ]:
import sys
from pathlib import Path

MODULE_NAMES = ("aws_credentials.py", "mcp_s3_client.py", "traffic_generator.py")


def _find_module_dir() -> Path | None:
    candidates = [
        Path.cwd(),
        Path.cwd() / "colab",
        Path("/content/traffic-generator/colab"),
        Path("/content/colab"),
    ]
    for directory in candidates:
        if all((directory / name).exists() for name in MODULE_NAMES):
            return directory.resolve()
    return None


module_dir = _find_module_dir()
if module_dir is None:
    print("Upload the helper modules from colab/:")
    for name in MODULE_NAMES:
        print(f"  • {name}")
    from google.colab import files

    uploaded = files.upload()
    module_dir = Path("/content/colab")
    module_dir.mkdir(parents=True, exist_ok=True)
    for name, content in uploaded.items():
        (module_dir / name).write_bytes(content)

    missing = [name for name in MODULE_NAMES if not (module_dir / name).exists()]
    if missing:
        raise SystemExit(f"Still missing: {', '.join(missing)}")

sys.path.insert(0, str(module_dir))
print(f"Using modules from: {module_dir}")
# Re-run friendly: drop cached helpers so Colab picks up file edits
for _name in list(sys.modules):
    if _name in ("aws_credentials", "mcp_s3_client", "traffic_generator"):
        del sys.modules[_name]



In [ ]:
# Import helpers (run before the traffic cell)
from aws_credentials import CredentialError, mask_access_key, resolve_aws_credentials
from mcp_s3_client import AwsMcpS3Client
from traffic_generator import run_traffic_loop, verify_access


## Run traffic

Adjust the settings below, then run the cell. You will see one line per operation (for example `PutObject -> OK`). When the loop finishes, the cell output includes a summary of operations and any errors.

| Setting | What it does |
|---------|----------------|
| **BUCKET** | S3 bucket InstantEvidence monitors |
| **CYCLES** | How many operations to run |
| **MIN_INTERVAL_SEC** / **MAX_INTERVAL_SEC** | Random pause between operations (seconds) |
| **READ_ONLY** | When enabled, only list/read/head operations run (no uploads, copies, or deletes) |


In [ ]:
BUCKET = "your-instantevidence-bucket"  # @param {type:"string"}
CYCLES = 20  # @param {type:"integer"}
MIN_INTERVAL_SEC = 3.0  # @param {type:"number"}
MAX_INTERVAL_SEC = 12.0  # @param {type:"number"}
READ_ONLY = False  # @param {type:"boolean"}

try:
    creds = resolve_aws_credentials()
    print(f"Using {mask_access_key(creds.access_key_id)} in {creds.region}")
except CredentialError as exc:
    raise SystemExit(exc) from exc

mcp = AwsMcpS3Client(creds, read_only=READ_ONLY)
stats = None
try:
    await mcp.__aenter__()
    await verify_access(mcp, BUCKET)
    print("Smoke test OK")
    stats = await run_traffic_loop(
        bucket=BUCKET,
        credentials=creds,
        client=mcp,
        cycles=CYCLES,
        min_interval_sec=MIN_INTERVAL_SEC,
        max_interval_sec=MAX_INTERVAL_SEC,
        read_only=READ_ONLY,
        verify=False,
    )
finally:
    await mcp.aclose()

stats


## View results in InstantEvidence

If your bucket is connected to InstantEvidence, you should see new S3 events in the console shortly after each operation completes. If nothing appears, confirm event delivery (SNS or EventBridge) is configured for the bucket you set in **BUCKET**.
